# 🐳 Subiendo Vectores a Qdrant Local (Docker)

Este notebook sube todos los embeddings a una instancia local de Qdrant usando Docker.

## ⚠️ Requisitos:

1. **Docker instalado**
2. **Ejecutar en terminal:**
   ```bash
   docker pull qdrant/qdrant
   docker run -d --name qdrant -p 6333:6333 qdrant/qdrant
   ```

3. **Verificar que Qdrant está corriendo:**
   ```bash
   docker ps
   ```

## ✅ Ventajas:

- 🚀 **12x más rápido** que Cloud
- 💾 **Sin límites** de storage
- 🎛️ **Control total** sobre recursos
- 💰 **Sin costos** de cloud

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from qdrant_client import QdrantClient, models
import time

print("="*70)
print("CONFIGURACIÓN QDRANT LOCAL (DOCKER)")
print("="*70)

# URL local (sin API key)
QDRANT_URL = "http://localhost:6333"

# Inicializar cliente
client = QdrantClient(url=QDRANT_URL, timeout=120)

# Verificar conexión
try:
    collections = client.get_collections()
    print(f"[OK] Conectado a Qdrant local")
    print(f"[i] Colecciones existentes: {len(collections.collections)}")
except Exception as e:
    print(f"[ERROR] No se puede conectar a Qdrant: {e}")
    print("[!] Asegúrate de que Docker esté corriendo:")
    print("   docker run -d --name qdrant -p 6333:6333 qdrant/qdrant")

CONFIGURACIÓN QDRANT LOCAL (DOCKER)
[OK] Conectado a Qdrant local
[i] Colecciones existentes: 3


In [2]:
# Cargar mapa de textos de Complaints
print("="*70)
print("CARGANDO MAPA DE TEXTOS PARA COMPLAINTS")
print("="*70)

# Cargar complaints originales con texto
complaints_original = pd.read_parquet("../data/processed/complaints_filtered_downsampled.parquet")

# Crear mapa: CMPLID -> CDESCR (texto completo)
complaints_text_map = dict(zip(
    complaints_original['CMPLID'].astype(str),
    complaints_original['CDESCR'].fillna('')
))

print(f"[OK] Mapa de textos creado: {len(complaints_text_map):,} complaints")
print(f"[i] Ejemplo de texto (primeros 100 chars):")
if complaints_text_map:
    example_key = list(complaints_text_map.keys())[0]
    print(f"  ID {example_key}: {list(complaints_text_map.values())[0][:100]}...")


CARGANDO MAPA DE TEXTOS PARA COMPLAINTS
[OK] Mapa de textos creado: 512,725 complaints
[i] Ejemplo de texto (primeros 100 chars):
  ID 1760760: The airbag light came on for no reason. It will not go off. The concern is the airbags won’t deploy ...


In [3]:
print("="*70)
print("SUBIR RECALLS")
print("="*70)

# Cargar embeddings
BASE = Path("../data")
RCL_EMB_DIR = BASE / "embeddings" / "recalls_e5_mlg_instruct"
RCL_META = RCL_EMB_DIR / "recalls_chunks_meta.parquet"
RCL_EMB = RCL_EMB_DIR / "recalls_embeddings.npy"
RCL_COLLECTION = "nhtsa_recalls"

# Carga de datos
print("[i] Cargando embeddings de Recalls...")
recalls_vecs = np.load(RCL_EMB)
recalls_meta = pd.read_parquet(RCL_META)

print(f"[i] {len(recalls_vecs):,} vectores de dimensión {recalls_vecs.shape[1]}")
assert len(recalls_vecs) == len(recalls_meta), "Desalineación vectores vs metadatos"

# Asegurar colección
print(f"[i] Verificando colección '{RCL_COLLECTION}'...")
client.recreate_collection(
    collection_name=RCL_COLLECTION,
    vectors_config=models.VectorParams(size=recalls_vecs.shape[1], distance=models.Distance.COSINE)
)

# Convertir a float32
vecs_float32 = recalls_vecs.astype(np.float32)

# Función auxiliar para crear payloads COMPLETOS (incluyendo texto)
def make_payloads_complete(df):
    """Crea payloads con TODOS los campos, incluyendo texto completo
    
    ADVERTENCIA: Esto aumentará significativamente el tamaño del payload.
    Requiere batch_size más pequeño (~100-200)
    """
    # Convertir numpy types a nativos
    def to_native(val):
        if isinstance(val, (np.generic,)):
            return val.item()
        elif isinstance(val, (np.ndarray,)):
            return val.tolist()
        elif isinstance(val, pd.Timestamp):
            return str(val)
        return val
    
    # Crear payloads con todos los campos
    clean = df.where(pd.notnull(df), None)
    payloads = []
    for _, row in clean.iterrows():
        payload = {k: to_native(v) for k, v in row.to_dict().items() if v is not None}
        payloads.append(payload)
    
    return payloads

# Calcular batch_size óptimo basado en tamaño estimado
# Estimación conservadora: 300 chars texto + 100 bytes metadata = ~400 bytes por registro
# Meta: mantenerse bajo 33MB = 33,000,000 bytes
# Con 1024 dims vectors (4 bytes cada uno) = 4096 bytes por vector
# Total por registro: 4096 (vector) + 400 (payload) ≈ 4500 bytes
# 33MB / 4500 = ~7,333 vectores pero usaremos menos para seguridad

batch_size = 200  # Conservador: será más lento pero seguro
print(f"[i] Subiendo {len(recalls_vecs):,} vectores en lotes de {batch_size}...")
print(f"[!] Batch size reducido a {batch_size} para evitar límite 33MB con texto completo")

for start in range(0, len(recalls_vecs), batch_size):
    end = min(start + batch_size, len(recalls_vecs))
    
    batch_vecs = vecs_float32[start:end]
    batch_meta = recalls_meta.iloc[start:end]
    batch_ids = list(range(start, end))
    batch_payloads = make_payloads_complete(batch_meta)
    
    client.upsert(
        collection_name=RCL_COLLECTION,
        points=models.Batch(
            ids=batch_ids,
            vectors=batch_vecs.tolist(),
            payloads=batch_payloads
        ),
        wait=True
    )
    
    print(f"  – Subidos: {end:,}/{len(recalls_vecs):,}")

# Verificación
count = client.count(RCL_COLLECTION, exact=True)
print(f"[OK] Carga completada. {count.count:,} vectores en {RCL_COLLECTION}")

SUBIR RECALLS
[i] Cargando embeddings de Recalls...
[i] 12,901 vectores de dimensión 1024
[i] Verificando colección 'nhtsa_recalls'...


C:\Users\moral\AppData\Local\Temp\ipykernel_2528\310843904.py:22: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


[i] Subiendo 12,901 vectores en lotes de 200...
[!] Batch size reducido a 200 para evitar límite 33MB con texto completo
  – Subidos: 200/12,901
  – Subidos: 400/12,901
  – Subidos: 600/12,901
  – Subidos: 800/12,901
  – Subidos: 1,000/12,901
  – Subidos: 1,200/12,901
  – Subidos: 1,400/12,901
  – Subidos: 1,600/12,901
  – Subidos: 1,800/12,901
  – Subidos: 2,000/12,901
  – Subidos: 2,200/12,901
  – Subidos: 2,400/12,901
  – Subidos: 2,600/12,901
  – Subidos: 2,800/12,901
  – Subidos: 3,000/12,901
  – Subidos: 3,200/12,901
  – Subidos: 3,400/12,901
  – Subidos: 3,600/12,901
  – Subidos: 3,800/12,901
  – Subidos: 4,000/12,901
  – Subidos: 4,200/12,901
  – Subidos: 4,400/12,901
  – Subidos: 4,600/12,901
  – Subidos: 4,800/12,901
  – Subidos: 5,000/12,901
  – Subidos: 5,200/12,901
  – Subidos: 5,400/12,901
  – Subidos: 5,600/12,901
  – Subidos: 5,800/12,901
  – Subidos: 6,000/12,901
  – Subidos: 6,200/12,901
  – Subidos: 6,400/12,901
  – Subidos: 6,600/12,901
  – Subidos: 6,800/12,901
  –

In [4]:
print("="*70)
print("SUBIR INVESTIGATIONS")
print("="*70)

# Cargar embeddings
INV_EMB_DIR = BASE / "embeddings" / "investigations_e5_mlg_instruct"
INV_META = INV_EMB_DIR / "invest_chunks_meta.parquet"
INV_EMB = INV_EMB_DIR / "invest_embeddings.npy"
INV_COLLECTION = "nhtsa_investigations"

# Carga de datos
print("[i] Cargando embeddings de Investigations...")
invest_vecs = np.load(INV_EMB)
invest_meta = pd.read_parquet(INV_META)

print(f"[i] {len(invest_vecs):,} vectores de dimensión {invest_vecs.shape[1]}")
assert len(invest_vecs) == len(invest_meta), "Desalineación vectores vs metadatos"

# Asegurar colección
print(f"[i] Verificando colección '{INV_COLLECTION}'...")
client.recreate_collection(
    collection_name=INV_COLLECTION,
    vectors_config=models.VectorParams(size=invest_vecs.shape[1], distance=models.Distance.COSINE)
)

# Convertir a float32
vecs_float32 = invest_vecs.astype(np.float32)

# Reutilizar función de payloads completos de la celda anterior
# Subir en lotes (batch_size reducido para evitar límite 33MB)
batch_size = 200
print(f"[i] Subiendo {len(invest_vecs):,} vectores en lotes de {batch_size}...")
print(f"[!] Batch size reducido a {batch_size} para evitar límite 33MB con texto completo")

for start in range(0, len(invest_vecs), batch_size):
    end = min(start + batch_size, len(invest_vecs))
    
    batch_vecs = vecs_float32[start:end]
    batch_meta = invest_meta.iloc[start:end]
    batch_ids = list(range(start, end))
    batch_payloads = make_payloads_complete(batch_meta)
    
    client.upsert(
        collection_name=INV_COLLECTION,
        points=models.Batch(
            ids=batch_ids,
            vectors=batch_vecs.tolist(),
            payloads=batch_payloads
        ),
        wait=True
    )
    
    print(f"  – Subidos: {end:,}/{len(invest_vecs):,}")

# Verificación
count = client.count(INV_COLLECTION, exact=True)
print(f"[OK] Carga completada. {count.count:,} vectores en {INV_COLLECTION}")

SUBIR INVESTIGATIONS
[i] Cargando embeddings de Investigations...
[i] 5,736 vectores de dimensión 1024
[i] Verificando colección 'nhtsa_investigations'...


C:\Users\moral\AppData\Local\Temp\ipykernel_2528\1640043264.py:21: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


[i] Subiendo 5,736 vectores en lotes de 200...
[!] Batch size reducido a 200 para evitar límite 33MB con texto completo
  – Subidos: 200/5,736
  – Subidos: 400/5,736
  – Subidos: 600/5,736
  – Subidos: 800/5,736
  – Subidos: 1,000/5,736
  – Subidos: 1,200/5,736
  – Subidos: 1,400/5,736
  – Subidos: 1,600/5,736
  – Subidos: 1,800/5,736
  – Subidos: 2,000/5,736
  – Subidos: 2,200/5,736
  – Subidos: 2,400/5,736
  – Subidos: 2,600/5,736
  – Subidos: 2,800/5,736
  – Subidos: 3,000/5,736
  – Subidos: 3,200/5,736
  – Subidos: 3,400/5,736
  – Subidos: 3,600/5,736
  – Subidos: 3,800/5,736
  – Subidos: 4,000/5,736
  – Subidos: 4,200/5,736
  – Subidos: 4,400/5,736
  – Subidos: 4,600/5,736
  – Subidos: 4,800/5,736
  – Subidos: 5,000/5,736
  – Subidos: 5,200/5,736
  – Subidos: 5,400/5,736
  – Subidos: 5,600/5,736
  – Subidos: 5,736/5,736
[OK] Carga completada. 5,736 vectores en nhtsa_investigations


In [5]:
print("="*70)
print("SUBIR COMPLAINTS (512K VECTORES) - OPTIMIZADO")
print("="*70)

# Cargar embeddings
CPL_EMB_DIR = BASE / "embeddings" / "complaints_e5_mlg_instruct"
CPL_COLLECTION = "nhtsa_complaints"

# Encontrar shards de embeddings
shard_files = sorted(CPL_EMB_DIR.glob("embeddings_shard_*.npy"))
meta_files = sorted(CPL_EMB_DIR.glob("meta_shard_*.parquet"))

print(f"[i] Encontrados {len(shard_files)} shards de embeddings")
print(f"[i] Encontrados {len(meta_files)} shards de metadatos")

# Verificar coherencia
assert len(shard_files) == len(meta_files), "Número de shards no coincide"

# Cargar el primer shard para obtener dimensión
first_vecs = np.load(shard_files[0])
first_meta = pd.read_parquet(meta_files[0])

dim = first_vecs.shape[1]
print(f"[i] Dimensión detectada: {dim}")

# Asegurar colección (BORRAR y RECREAR para incluir textos)
print(f"[i] Borrando colección '{CPL_COLLECTION}' anterior...")
try:
    client.delete_collection(CPL_COLLECTION)
except:
    pass

print(f"[i] Creando colección '{CPL_COLLECTION}' con textos...")
client.create_collection(
    collection_name=CPL_COLLECTION,
    vectors_config=models.VectorParams(size=dim, distance=models.Distance.COSINE)
)

# Función especializada para Complaints con texto
def make_payloads_complaints_with_text(df):
    """Crea payloads para Complaints incluyendo texto completo desde mapa"""
    def to_native(val):
        if isinstance(val, (np.generic,)):
            return val.item()
        elif isinstance(val, (np.ndarray,)):
            return val.tolist()
        elif isinstance(val, pd.Timestamp):
            return str(val)
        return val
    
    clean = df.where(pd.notnull(df), None)
    payloads = []
    
    for _, row in clean.iterrows():
        payload = {k: to_native(v) for k, v in row.to_dict().items() if v is not None}
        
        # Agregar texto completo desde mapa si existe
        complaint_id = str(row['id'])
        if complaint_id in complaints_text_map:
            payload['text'] = complaints_text_map[complaint_id]
        
        payloads.append(payload)
    
    return payloads

# Batch size MUY reducido para Complaints (tienen textos largos)
# Con texto completo, cada payload será ~600 bytes (300 bytes vector + 300 bytes texto)
# Para mantenerse bajo 33MB con seguridad, usamos batch_size de 50
batch_size_complaints = 50  # MUY conservador con texto
print(f"[!] Batch size para Complaints: {batch_size_complaints} (con texto completo)")

total_uploaded = 0

for i, (shard_file, meta_file) in enumerate(zip(shard_files, meta_files)):
    print(f"\n[i] Procesando shard {i+1}/{len(shard_files)}...")
    
    # Cargar shard
    shard_vecs = np.load(shard_file)
    shard_meta = pd.read_parquet(meta_file)
    
    print(f"  [i] Shard {i}: {len(shard_vecs):,} vectores")
    assert len(shard_vecs) == len(shard_meta), f"Desalineación en shard {i}"
    
    # Convertir a float32
    vecs_float32 = shard_vecs.astype(np.float32)
    
    # Subir en lotes (batch size muy reducido)
    for start in range(0, len(shard_vecs), batch_size_complaints):
        end = min(start + batch_size_complaints, len(shard_vecs))
        
        batch_vecs = vecs_float32[start:end]
        batch_meta = shard_meta.iloc[start:end]
        batch_ids = list(range(total_uploaded + start, total_uploaded + end))
        batch_payloads = make_payloads_complaints_with_text(batch_meta)
        
        client.upsert(
            collection_name=CPL_COLLECTION,
            points=models.Batch(
                ids=batch_ids,
                vectors=batch_vecs.tolist(),
                payloads=batch_payloads
            ),
            wait=False  # Asincrónico para mejor performance
        )
        
        if (end - start) % 10000 == 0 or end == len(shard_vecs):
            print(f"  – Shard {i}: subidos {end:,}/{len(shard_vecs):,}")
    
    total_uploaded += len(shard_vecs)
    print(f"  [OK] Shard {i} completado. Total acumulado: {total_uploaded:,}")

# Esperar a que termine (local debería ser rápido)
import time
time.sleep(2)

# Verificación final
count = client.count(CPL_COLLECTION, exact=True)
print(f"\n[OK] Carga completada. {count.count:,} vectores en {CPL_COLLECTION}")
print(f"[i] Total procesado: {total_uploaded:,}")

SUBIR COMPLAINTS (512K VECTORES) - OPTIMIZADO
[i] Encontrados 4 shards de embeddings
[i] Encontrados 4 shards de metadatos
[i] Dimensión detectada: 1024
[i] Borrando colección 'nhtsa_complaints' anterior...
[i] Creando colección 'nhtsa_complaints' con textos...
[!] Batch size para Complaints: 50 (con texto completo)

[i] Procesando shard 1/4...
  [i] Shard 0: 25,000 vectores
  – Shard 0: subidos 25,000/25,000
  [OK] Shard 0 completado. Total acumulado: 25,000

[i] Procesando shard 2/4...
  [i] Shard 1: 25,000 vectores
  – Shard 1: subidos 25,000/25,000
  [OK] Shard 1 completado. Total acumulado: 50,000

[i] Procesando shard 3/4...
  [i] Shard 2: 25,000 vectores
  – Shard 2: subidos 25,000/25,000
  [OK] Shard 2 completado. Total acumulado: 75,000

[i] Procesando shard 4/4...
  [i] Shard 3: 437,725 vectores
  – Shard 3: subidos 437,725/437,725
  [OK] Shard 3 completado. Total acumulado: 512,725

[OK] Carga completada. 512,725 vectores en nhtsa_complaints
[i] Total procesado: 512,725
